# Fundamentals of Machine Learning — Exercise 11
## Regression on Three Real Datasets and Project Part 2 Preparation

The last modelling exercise before the project defences. You compare regression approaches on **three real-world datasets** and practise exactly what Project Part 2 asks for: a baseline, a fair model comparison, no leakage and a look at the errors.

| Dataset | Rows × features | Target |
|---|---:|---|
| Petrol consumption | 48 × 4 | petrol consumption per state |
| Diabetes | 442 × 10 | disease progression after one year |
| House Prices | about 1460 × 6 | sale price |

### Plan 
| Part | Topic |  |
|---|---|---:|
| 1 | Metrics and baseline |  |
| 2 | Models in a pipeline |  |
| 3 | Cross-validation and comparison (Petrol) | |
| 4 | The same code on three datasets |  |
| 5 | Error analysis | |

Optional extensions at the end (one feature vs four, Ridge coefficients, tuning `alpha`, Lasso, trees and extrapolation, log target) are for home or for spare time.

### Symbols
| | |
|---|---|
| 🔮 **Predict** | write your guess **before** running the cell |
| 🛠️ **Do** | a short `TODO`; the next cell checks it (hints are collapsed) |
| 🕵️ **Think** | answer in your own words; the answer checks are at the very end |

### Rule of the day
Models are compared with the **same** 5-fold cross-validation. In your project, first hold out a test set, use cross-validation only on the training part for selection and tuning, and evaluate the test set **once** at the end. Imputing, scaling, encoding and feature selection belong **inside** a `Pipeline`.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.compose import TransformedTargetRegressor
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, cross_val_score, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": True, "grid.alpha": .3})


def check(test, ok="Correct.", hint="Not yet."):
    """Self-check: `test` is a function returning True/False (an error counts as a failure)."""
    try:
        passed = bool(test())
    except Exception as e:
        passed = False
        hint = hint + f"  [{type(e).__name__}: {e}]"
    print(("✅ " + ok) if passed else ("❌ " + hint))


# ---------- Dataset 1: Petrol consumption (embedded) ----------
PETROL_CSV = r"""Petrol_tax,Average_income,Paved_Highways,Population_Driver_licence(%),Petrol_Consumption
9,3571,1976,0.525,541
9,4092,1250,0.572,524
9,3865,1586,0.58,561
7.5,4870,2351,0.529,414
8,4399,431,0.544,410
10,5342,1333,0.571,457
8,5319,11868,0.451,344
8,5126,2138,0.553,467
8,4447,8577,0.529,464
7,4512,8507,0.552,498
8,4391,5939,0.53,580
7.5,5126,14186,0.525,471
7,4817,6930,0.574,525
7,4207,6580,0.545,508
7,4332,8159,0.608,566
7,4318,10340,0.586,635
7,4206,8508,0.572,603
7,3718,4725,0.54,714
7,4716,5915,0.724,865
8.5,4341,6010,0.677,640
7,4593,7834,0.663,649
8,4983,602,0.602,540
9,4897,2449,0.511,464
9,4258,4686,0.517,547
8.5,4574,2619,0.551,460
9,3721,4746,0.544,566
8,3448,5399,0.548,577
7.5,3846,9061,0.579,631
8,4188,5975,0.563,574
9,3601,4650,0.493,534
7,3640,6905,0.518,571
7,3333,6594,0.513,554
8,3063,6524,0.578,577
7.5,3357,4121,0.547,628
8,3528,3495,0.487,487
6.58,3802,7834,0.629,644
5,4045,17782,0.566,640
7,3897,6385,0.586,704
8.5,3635,3274,0.663,648
7,4345,3905,0.672,968
7,4449,4639,0.626,587
7,3656,3985,0.563,699
7,4300,3635,0.603,632
7,3745,2611,0.508,591
6,5215,2302,0.672,782
9,4476,3942,0.571,510
7,4296,4083,0.623,610
7,5002,9794,0.593,524""" #"""
petrol = pd.read_csv(StringIO(PETROL_CSV))
X_petrol, y_petrol = petrol.drop(columns="Petrol_Consumption"), petrol["Petrol_Consumption"]

# ---------- Dataset 2: Diabetes (built in) ----------
diab = load_diabetes(as_frame=True)
X_diab, y_diab = diab.data, diab.target

# ---------- Dataset 3: House Prices (course data, downloaded) ----------
HOUSE_URL = ("https://raw.githubusercontent.com/rasvob/VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/"
             "master/datasets/zsu_cv1_data.csv")
HOUSE_FEATURES = ["OverallQual", "OverallCond", "YearBuilt", "GrLivArea", "LotArea", "BedroomAbvGr"]
try:
    house = pd.read_csv(HOUSE_URL)
    X_house, y_house = house[HOUSE_FEATURES].copy(), house["SalePrice"].copy()
    print("House Prices loaded:", X_house.shape)
except Exception as e:
    house = X_house = y_house = None
    print("⚠️ House Prices could not be downloaded (" + type(e).__name__ + "). Parts that need it will be skipped; "
          "run the notebook again with internet access (e.g. in Colab).")

datasets = {"Petrol": (X_petrol, y_petrol), "Diabetes": (X_diab, y_diab)}
if house is not None:
    datasets["House Prices"] = (X_house, y_house)
print({k: v[0].shape for k, v in datasets.items()})

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
SCORING = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

def cv_table(models, X, y):
    """Cross-validated MAE, RMSE, R² and the spread of RMSE over folds, best RMSE first."""
    rows = {}
    for name, m in models.items():
        s = cross_validate(m, X, y, cv=cv, scoring=SCORING)
        rows[name] = {"MAE": -s["test_MAE"].mean(), "RMSE": -s["test_RMSE"].mean(), "R2": s["test_R2"].mean(), "RMSE std": s["test_RMSE"].std()}
    return pd.DataFrame(rows).T.sort_values("RMSE")

---
# 1. Metrics and baseline 

> 🧠 **Key idea:** **MAE** = the typical size of an error (units of the target). **RMSE** = the same, but squaring makes **large errors count more**. **R²** compares the model with the **baseline "always predict the mean"**: 0 = as good as the mean, **negative = worse than the mean**. Every model must beat the baseline.

Four houses, true value 100 each. Model A makes four errors of size 10, model B one error of size 20.

In [ ]:
y_demo = np.array([100., 100., 100., 100.])
pred_a = np.array([90., 90., 110., 110.])
pred_b = np.array([100., 100., 100., 120.])

### 🔮 Predict
Before computing anything: which model has the **larger RMSE** and which the **larger MAE**? Write `"A"`, `"B"` or `"same"`.

In [ ]:
guess_rmse = None    # TODO: "A", "B" or "same": which model has the larger RMSE?
guess_mae = None     # TODO: "A", "B" or "same": which model has the larger MAE?

In [ ]:
for name, p in [("A", pred_a), ("B", pred_b)]:
    print(name, " MAE", mean_absolute_error(y_demo, p), " RMSE", mean_squared_error(y_demo, p) ** 0.5)
print("your guesses: RMSE", guess_rmse, " MAE", guess_mae)

### 🛠️ The baseline
The reference for every regression model is "always predict the mean". Evaluate it on Petrol with the same cross-validation.

In [ ]:
baseline = DummyRegressor(strategy="mean")
base_scores = None   # TODO: cross_validate(baseline, X_petrol, y_petrol, cv=cv, scoring=SCORING)
print("baseline on Petrol: RMSE %.1f, R2 %.2f" % (-base_scores["test_RMSE"].mean(), base_scores["test_R2"].mean()))

<details><summary>💡 Hint</summary>

`cross_validate(model, X, y, cv=cv, scoring=SCORING)` returns a dictionary; the metrics are under `test_MAE`, `test_RMSE`, `test_R2` (the first two are negated by scikit-learn).

</details>

In [ ]:
check(lambda: guess_rmse in ("A", "B", "same") and guess_mae in ("A", "B", "same") and "test_RMSE" in base_scores, "Guesses written and the baseline evaluated.", "Fill in both guesses and the cross_validate call.")

**Question 1 🕵️.** Why are the two RMSEs equal although the MAEs are not? Which metric would you choose for (a) parcel delivery, where a delay of 1 hour is about twice as bad as 30 minutes, and (b) a bridge-load estimate, where one large underestimate can be catastrophic?

**Question 2 🕵️.** What does the baseline predict, what does a **negative** R² mean, and why must a real model beat the baseline?

---
# 2. Models in a pipeline 

> 🧠 **Key idea:** many common model families have both classification and regression versions. A **Ridge** (linear regression with a penalty on large coefficients) should generally be used with **scaled** features when the features have different scales, because the penalty acts on coefficients measured in different units unequally. So we put `StandardScaler` and `Ridge` into one `Pipeline`: the scaler is then fitted again on the training part of every fold. A **random forest** averages many trees; **gradient boosting** builds trees one after another, each correcting the previous ones. Trees do not need scaling.

### 🔮 Predict
Which model will have the lowest RMSE on Petrol (48 rows)? `"ridge"`, `"random forest"` or `"gradient boosting"`.

In [ ]:
guess_petrol = None      # TODO: your guess

petrol_models = {
    "baseline (mean)": DummyRegressor(strategy="mean"),
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge": None,             # TODO: a pipeline StandardScaler -> Ridge(alpha=10)
    "random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE),
    "gradient boosting": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=7, min_samples_leaf=4, random_state=RANDOM_STATE),
}

<details><summary>💡 Hint</summary>

`make_pipeline(StandardScaler(), Ridge(alpha=10))`: first the scaler, then the model.

</details>

In [ ]:
check(lambda: guess_petrol is not None and petrol_models["ridge"] is not None and len(petrol_models["ridge"].steps) == 2, "Ridge is a two-step pipeline: scaler, then Ridge.", "Fill in the guess and the Ridge pipeline.")

**Question 3 🕵️.** Why does the Ridge model include `StandardScaler`, while the random forest does not? Why is the scaler placed inside the pipeline instead of scaling all the data first?

---
# 3. Cross-validation and comparison on Petrol 

> 🧠 **Key idea:** a single number is not enough. We compare models on the **same folds** and look at the **mean** and at the **spread** (`RMSE std`) over folds. If the difference between the mean scores is small compared with the fold-to-fold variability, the evidence for declaring one model better is weak.

In [ ]:
petrol_cv = cv_table(petrol_models, X_petrol, y_petrol)
display(petrol_cv.round(2))
lead = petrol_cv["RMSE"].iloc[1] - petrol_cv["RMSE"].iloc[0]     # 2nd best minus best
spread = petrol_cv["RMSE std"].iloc[0]                            # fold-to-fold std of the best model
print("your guess:", guess_petrol, "  best:", petrol_cv.index[0], "  lead %.1f  vs  fold spread %.1f" % (lead, spread))

**Question 4 🕵️.** Is the difference between the two best models large compared with the fold-to-fold variability? With only 48 rows, why is it risky to announce a "winner"?

---
# 4. The same code on three datasets 

> 🧠 **Key idea:** the best model depends on the data (size, noise, non-linearity). MAE and RMSE are in the units of each target (consumption, a disease score, dollars), so they are **not comparable across datasets**: compare models **within** a dataset, and against its baseline.

The evaluator below is given: the same protocol (imputing inside the pipeline, the same 5-fold CV) for any dataset.

In [ ]:
def compare_regressors(X, y, name):
    imp = lambda: SimpleImputer(strategy="median")
    models = {
        "baseline (mean)": make_pipeline(imp(), DummyRegressor()),
        "ridge": make_pipeline(imp(), StandardScaler(), Ridge(alpha=10)),
        "random forest": make_pipeline(imp(), RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE)),
        "gradient boosting": make_pipeline(imp(), HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=5, random_state=RANDOM_STATE)),
    }
    out = []
    for mname, m in models.items():
        s = cross_validate(m, X, y, cv=cv, scoring=SCORING)
        out.append({"dataset": name, "model": mname, "MAE": -s["test_MAE"].mean(), "RMSE": -s["test_RMSE"].mean(),
                    "R2": s["test_R2"].mean(), "R2 std": s["test_R2"].std()})
    return pd.DataFrame(out)

### 🔮 Predict
For each dataset guess the model with the lowest RMSE (`"ridge"`, `"random forest"` or `"gradient boosting"`).

In [ ]:
pred_winner = {"Petrol": None, "Diabetes": None, "House Prices": None}     # TODO: your guesses

all_results = pd.concat([compare_regressors(X, y, name) for name, (X, y) in datasets.items()], ignore_index=True)
display(all_results.round(3))
actual_winner = all_results[all_results["model"] != "baseline (mean)"].loc[lambda d: d.groupby("dataset")["RMSE"].idxmin(), ["dataset", "model"]].set_index("dataset")["model"].to_dict()
print("your guesses:", pred_winner); print("actual:      ", actual_winner)

In [ ]:
check(lambda: all(v is not None for v in pred_winner.values()) and len(all_results) == 4 * len(datasets), "Guesses written; all models evaluated on all datasets.", "Fill in a guess for every dataset.")

**Question 5 🕵️.** Does the same model perform best on all three datasets? What does this tell us about model selection? Why should raw RMSE values not be compared across datasets?

---
# 5. Error analysis 

> 🧠 **Key idea:** a score does not show *where* the model fails. Plot the **residuals** against the predictions and look at the largest errors. Residual = observed − predicted: positive = the model under-predicted, negative = over-predicted. Random scatter around 0 is healthy; a curve or a funnel is a warning.

We use **out-of-fold** predictions: each row was predicted by a model that did not train on it.

In [ ]:
pipe_d = make_pipeline(StandardScaler(), Ridge(alpha=10))
pred_d = cross_val_predict(pipe_d, X_diab, y_diab, cv=cv)
resid_d = None            # TODO: observed minus predicted (y_diab.values and pred_d)
worst = np.argsort(np.abs(resid_d))[-5:]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(pred_d, resid_d, s=8); ax[0].axhline(0, c="k", ls="--"); ax[0].set(xlabel="predicted", ylabel="residual")
ax[1].hist(resid_d, bins=30); ax[1].set(title="distribution of residuals")
plt.tight_layout(); plt.show()
print("worst five: observed", y_diab.values[worst].round(0), " predicted", pred_d[worst].round(0))

<details><summary>💡 Hint</summary>

`resid_d = y_diab.values - pred_d`

</details>

In [ ]:
check(lambda: len(resid_d) == len(y_diab) and abs(resid_d.mean()) < 5, "Residuals are centred near 0.", "Residual = observed − predicted.")

**Question 6 🕵️.** Are the residuals randomly scattered around zero? Where does the model make the largest errors? What would you investigate next?

---
### Answer checks (compare with your own answers)

<details><summary>Question 1: MAE vs RMSE</summary>

Model A has four errors of size 10 (MAE = RMSE = 10). Model B has one error of 20 and three zeros: MAE = 5, but RMSE = √(400/4) = 10. RMSE squares errors, so one large error counts as much as four moderate ones; MAE counts every error linearly.
(a) Parcel delivery, where the cost grows in proportion to the delay: **MAE**. (b) Bridge load, where a single large underestimate is catastrophic: **RMSE** (or an asymmetric loss).

</details>

<details><summary>Question 2: baseline and R²</summary>

The baseline predicts the training mean for every row. R² = 0 means "as good as the mean", negative R² means worse than the mean (on Petrol the baseline has R² ≈ −0.24 in cross-validation, because each fold's own mean differs from the test fold's mean). A model that cannot beat this baseline has learned nothing useful, so every comparison starts from it.

</details>

<details><summary>Question 3: scaler in Ridge but not in the forest</summary>

Ridge penalises the size of the coefficients, and the size depends on the units of each feature, so Ridge should generally be used with scaled features when they have different scales. Trees only compare a feature with thresholds, so rescaling does not change them. The scaler is inside the pipeline so that it is fitted again on the training part of every fold; scaling all the data first would let information from the validation rows leak into the model.

</details>

<details><summary>Question 4: lead versus fold variability</summary>

Ridge has the best mean RMSE (≈ 65.9), only about 0.9 ahead of plain linear regression (≈ 66.8), while the fold-to-fold spread is about 18–27. The lead is far smaller than the variability, so the evidence for declaring one model better is weak. With 48 rows each fold has about 10 states and scores vary a lot, so report the mean and spread, call the top models comparable and prefer the simplest. All real models clearly beat the baseline (≈ 106).

</details>

<details><summary>Question 5: three datasets</summary>

No model is guaranteed to win everywhere. On Petrol and Diabetes the regularised linear model has the lowest RMSE (the result is consistent with a mostly linear signal; the more complex models do not provide an improvement under this evaluation). On the larger House Prices data tree ensembles are expected to win (check your own table). So model selection must be repeated for each problem, never copied from another one. Raw RMSE is in the units of the target (consumption, a disease score, dollars), so values for different datasets cannot be compared.

</details>

<details><summary>Question 6: residuals on Diabetes</summary>

The residuals are centred near 0, suggesting little overall mean bias; however, patterns in the residual plot can still reveal systematic errors. The largest errors are about ±130–155 and occur in both directions (e.g. observed 68, predicted 201; observed 311, predicted 175). The spread is somewhat larger for high predictions (a mild funnel, which the teacher can point out). Next: non-linear terms or interactions, tree ensembles, a target transformation, or accepting the limit of the 10 features (R² ≈ 0.48).

</details>

---
# Optional extensions (home or spare time)

## A. One feature or four; Ridge coefficients and scaling
One fixed split is used only to compare and draw. The raw-versus-scaled Ridge shows why scaling matters: a feature on a small scale (the share of driver licences, 0.45–0.7) needs a huge coefficient and is penalised much more.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_petrol, y_petrol, test_size=0.25, random_state=RANDOM_STATE)
one = "Population_Driver_licence(%)"
lin_one = LinearRegression().fit(Xtr[[one]], ytr); lin_all = LinearRegression().fit(Xtr, ytr)
print("test RMSE: one feature %.1f   four features %.1f" % (mean_squared_error(yte, lin_one.predict(Xte[[one]])) ** .5, mean_squared_error(yte, lin_all.predict(Xte)) ** .5))
ridge_raw = Ridge(alpha=10).fit(Xtr, ytr); ridge_scaled = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xtr, ytr)
display(pd.DataFrame({"linear (raw)": lin_all.coef_, "Ridge alpha=10 (raw)": ridge_raw.coef_, "Ridge alpha=10 (scaled)": ridge_scaled[-1].coef_}, index=Xtr.columns).round(3))

**Optional question A.** Which raw coefficient was shrunk the most and why? Only 12 states are in the test part: how much do you trust the one-vs-four comparison?

## B. Choosing `alpha` by cross-validation

In [ ]:
for a in [0.01, 1, 10, 100, 1000]:
    s = cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=a)), X_petrol, y_petrol, cv=cv, scoring="neg_root_mean_squared_error")
    print("alpha", a, " CV RMSE %.1f" % -s.mean())

**Optional question B.** What happens for a tiny and for a huge `alpha`? Is the best `alpha` clearly better than its neighbours?

## C. Lasso sets coefficients to exactly zero

In [ ]:
for a in [0.1, 1, 5, 10, 20, 50]:
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=10000)).fit(Xtr, ytr)
    print("alpha", a, " zero coefficients:", int(np.sum(lasso[-1].coef_ == 0)))

## D. Trees cannot extrapolate

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 300); y = 3 * x + rng.normal(0, 1, 300)
tr, te = x < 7, x >= 7
lin = LinearRegression().fit(x[tr, None], y[tr]); rf = RandomForestRegressor(100, random_state=0).fit(x[tr, None], y[tr])
grid = np.linspace(0, 10, 200)[:, None]
plt.scatter(x[tr], y[tr], s=8, label="train (x < 7)"); plt.scatter(x[te], y[te], s=8, c="gray", label="test (x ≥ 7)")
plt.plot(grid, lin.predict(grid), label="linear"); plt.plot(grid, rf.predict(grid), label="random forest"); plt.legend(); plt.show()
print("test RMSE: linear %.2f  random forest %.2f" % (mean_squared_error(y[te], lin.predict(x[te, None])) ** .5, mean_squared_error(y[te], rf.predict(x[te, None])) ** .5))

**Optional question D.** What does the forest predict for `x > 7` and when can this hurt in a real project?

## E. Target transformation (needs House Prices)
Training on `log(price)` changes the emphasis from absolute to relative differences. It is not guaranteed to improve every metric.

In [ ]:
if house is None:
    print("House Prices not available: skip this cell.")
else:
    rf_plain = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE))
    rf_log = TransformedTargetRegressor(regressor=rf_plain, func=np.log1p, inverse_func=np.expm1)
    rows = []
    for name, m in [("original price", rf_plain), ("log(price)", rf_log)]:
        s = cross_validate(m, X_house, y_house, cv=cv, scoring=SCORING)
        rows.append({"target": name, "MAE": -s["test_MAE"].mean(), "RMSE": -s["test_RMSE"].mean(), "R2": s["test_R2"].mean()})
    display(pd.DataFrame(rows).round(3))

**Optional question E.** Did the log transform improve every metric? Why can MAE and RMSE disagree?

## F. Normalised RMSE (nRMSE)
We use nRMSE only to express the error relative to the variability of each target: `RMSE / std(y)`. A mean baseline usually has nRMSE around 1, smaller is better. You do not necessarily need it in your project.

In [ ]:
all_results["nRMSE"] = all_results["RMSE"] / all_results["dataset"].map({name: y.std() for name, (X, y) in datasets.items()})
display(all_results.pivot(index="model", columns="dataset", values="nRMSE").round(2))